# DATA COLLECTION (RELIABLE)

# **1. TVNET**

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlsplit, urlunsplit

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "lv,en;q=0.9",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

REQUEST_SLEEP = 0.8

SECTIONS = [
    {"topic": "TVNET section 4237", "start_url": "https://www.tvnet.lv/section/4237", "target_n": 100},
    {"topic": "TVNET section 4282", "start_url": "https://www.tvnet.lv/section/4282", "target_n": 100},
    {"topic": "TVNET section 4211", "start_url": "https://www.tvnet.lv/section/4211", "target_n": 100},
    {"topic": "TVNET section 4230", "start_url": "https://www.tvnet.lv/section/4230", "target_n": 100},
    {"topic": "TVNET section 4292", "start_url": "https://www.tvnet.lv/section/4292", "target_n": 100},
]

SOURCE_NAME = "TVNET"
CLASS_LABEL = 0  # reliable/news class (adjust if you need)

def normalize_ws(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def canonical_tvnet_url(url: str) -> str:
    """
    Deduplicate /comments and remove tracking/query fragments.
    Keeps scheme+netloc+path.
    """
    parts = urlsplit(url)
    path = parts.path
    if path.endswith("/comments"):
        path = path[:-len("/comments")]
    return urlunsplit((parts.scheme, parts.netloc, path, "", ""))

def build_section_page_url(start_url: str, page: int) -> str:
    # TVNET sections typically paginate as /page/<n>/
    if page <= 1:
        return start_url
    if start_url.endswith("/"):
        return f"{start_url}page/{page}/"
    return f"{start_url}/page/{page}/"

def extract_section_article_urls(html: str) -> list[str]:
    """
    Extract article URLs from a section listing page.
    We keep URLs that look like TVNET articles: /<digits>/<slug>
    """
    soup = BeautifulSoup(html, "html.parser")
    urls = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href:
            continue

        if href.startswith("/"):
            href = "https://www.tvnet.lv" + href

        if not href.startswith("https://www.tvnet.lv/"):
            continue

        # Exclude section pages or non-article pages
        if "/section/" in href or "/term/" in href:
            continue

        # Article pattern: https://www.tvnet.lv/<id>/<slug>
        # Sometimes additional path parts exist; accept if it starts with /<digits>/
        path = urlsplit(href).path
        if not re.match(r"^/\d{6,}/", path):
            continue

        # Remove /comments variant here already
        href = canonical_tvnet_url(href)

        urls.add(href)

    return sorted(urls)

def discover_n_urls(start_url: str, n: int, max_pages: int = 200) -> list[str]:
    collected = []
    seen = set()
    empty_pages_in_a_row = 0

    for page in range(1, max_pages + 1):
        page_url = build_section_page_url(start_url, page)
        r = SESSION.get(page_url, timeout=30)
        if r.status_code == 404:
            print(f"Page {page:03d}: status 404 (stop)")
            break
        r.raise_for_status()

        page_urls = extract_section_article_urls(r.text)

        added = 0
        for u in page_urls:
            cu = canonical_tvnet_url(u)
            if cu not in seen:
                seen.add(cu)
                collected.append(cu)
                added += 1
                if len(collected) >= n:
                    return collected[:n]

        if added == 0:
            empty_pages_in_a_row += 1
        else:
            empty_pages_in_a_row = 0

        print(f"Page {page:03d}: found {len(page_urls)}; added {added}; total {len(collected)}")

        if empty_pages_in_a_row >= 5:
            break

        time.sleep(REQUEST_SLEEP)

    return collected[:n]

def extract_pub_date(soup: BeautifulSoup) -> str:
    # Try meta tags first
    for attrs in [
        {"property": "article:published_time"},
        {"name": "article:published_time"},
        {"itemprop": "datePublished"},
    ]:
        tag = soup.find("meta", attrs=attrs)
        if tag and tag.get("content"):
            c = tag["content"].strip()
            m = re.search(r"(\d{4}-\d{2}-\d{2})", c)
            if m:
                return m.group(1)

    # Common TVNET pattern: <div itemprop="datePublished" content="YYYY-MM-DD...">
    date_div = soup.find("div", attrs={"itemprop": "datePublished"})
    if date_div and date_div.get("content"):
        c = date_div["content"].strip()
        m = re.search(r"(\d{4}-\d{2}-\d{2})", c)
        if m:
            return m.group(1)

    return ""

def extract_title(soup: BeautifulSoup) -> str:
    # TVNET often: h1.article__headline
    h1 = soup.find("h1", class_="article__headline") or soup.find("h1")
    if h1:
        return normalize_ws(h1.get_text(" ", strip=True))
    if soup.title:
        return normalize_ws(soup.title.get_text(" ", strip=True))
    return ""

def extract_full_text(soup: BeautifulSoup) -> str:
    # TVNET body chunks
    content_divs = soup.find_all("div", class_="article-body__item--htmlElement")
    parts = []

    if content_divs:
        for d in content_divs:
            t = d.get_text(" ", strip=True)
            if t:
                parts.append(t)
    else:
        # Fallback: within article tag paragraphs
        article = soup.find("article")
        ps = article.find_all("p") if article else soup.find_all("p")
        for p in ps:
            t = p.get_text(" ", strip=True)
            if t:
                parts.append(t)

    txt = normalize_ws(" ".join(parts))

    # Optional: cut obvious TVNET boilerplate tails if present
    tail_markers = [
        "TVNET+",
        "TVNET ZIŅAS",
        "TVNET PLANĒTA",
        "TVNET EGOISTE",
    ]
    for m in tail_markers:
        idx = txt.find(m)
        if idx != -1 and idx > 0:
            txt = normalize_ws(txt[:idx])
            break

    return txt

def scrape_article(url: str, topic: str) -> dict:
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    title = extract_title(soup)
    pub_date = extract_pub_date(soup)
    full_text = extract_full_text(soup)

    return {
        "title": title,
        "full_text": full_text,
        "source": SOURCE_NAME,
        "publication_date": pub_date,
        "topic": topic,
        "class_label": int(CLASS_LABEL),
        "url": url,
        "full_text_original": full_text,
        "text_length": len(full_text),
    }

all_dfs = []

for sec in SECTIONS:
    print("\n======================================")
    print(f"Discovering URLs for: {sec['topic']}")
    urls = discover_n_urls(sec["start_url"], sec["target_n"], max_pages=300)
    print(f"Discovered {len(urls)} URLs for {sec['topic']}")

    data = []
    for i, u in enumerate(urls, 1):
        try:
            print(f"Scraping {i}/{len(urls)}: {u}")
            rec = scrape_article(u, sec["topic"])
            data.append(rec)
        except Exception as e:
            print(f"Error: {u} -> {e}")
        time.sleep(REQUEST_SLEEP)

    df = pd.DataFrame(
        data,
        columns=[
            "title",
            "full_text",
            "source",
            "publication_date",
            "topic",
            "class_label",
            "url",
            "full_text_original",
            "text_length",
        ],
    )

    out_name = f"tvnet_{urlsplit(sec['start_url']).path.strip('/').replace('/','_')}_{sec['target_n']}.csv"
    df.to_csv(out_name, index=False, encoding="utf-8")
    print(f"Saved: {out_name} | rows: {len(df)}")

    all_dfs.append(df)

merged = pd.concat(all_dfs, ignore_index=True)
merged.to_csv("tvnet_reliable_raw.csv", index=False, encoding="utf-8")
print("\nSaved merged dataset: vnet_reliable_raw.csv.csv")
print("Rows:", len(merged))
print("Per-topic counts:\n", merged["topic"].value_counts())


Discovering URLs for: TVNET section 4237
Page 001: found 36; added 36; total 36
Page 002: status 404 (stop)
Discovered 36 URLs for TVNET section 4237
Scraping 1/36: https://www.tvnet.lv/7808525/pirmdiena-bija-karstaka-diena-pasaule-kops-merijumu-saksanas
Scraping 2/36: https://www.tvnet.lv/7808551/ja-zeme-attalinas-no-saules-tad-kapec-ir-tik-karsts
Scraping 3/36: https://www.tvnet.lv/7809558/video-eiropu-sasper-tukstosiem-zibens-sautru
Scraping 4/36: https://www.tvnet.lv/7812584/pagajusi-vasara-nogalinaja-60-tukstosus-eiropiesu
Scraping 5/36: https://www.tvnet.lv/7812922/geologi-1950-gada-sakas-cilveka-laikmets
Scraping 6/36: https://www.tvnet.lv/7815847/naves-ieleja-somenes-varetu-tikt-parspets-pasaules-karstuma-rekords
Scraping 7/36: https://www.tvnet.lv/7818628/karstuma-vilnis-eiropa-jeb-nedela-no-elles
Scraping 8/36: https://www.tvnet.lv/7820250/atakamas-tuksnesis-saulainaka-vieta-uz-planetas
Scraping 9/36: https://www.tvnet.lv/7820545/dazi-miti-par-ledus-laikmetu-vai-mes-tada-dzi

# 1.1. raw data inspection

In [ ]:
import pandas as pd
import re
from collections import Counter

FILE = "tvnet_reliable_raw.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# -----------------------------
# TITLE INSPECTION
# -----------------------------
print("\nFIRST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).head(5), start=1):
        print(f"{i}. {title}")

print("\nLAST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).tail(5), start=1):
        print(f"{i}. {title}")

# -----------------------------
# TEXT LENGTH STATS
# -----------------------------
print("\nTEXT LENGTH SUMMARY:")
if "full_text" in df.columns:
    lengths = df["full_text"].fillna("").astype(str).str.len()
    print("Min:", int(lengths.min()))
    print("Median:", int(lengths.median()))
    print("Max:", int(lengths.max()))

# -----------------------------
# REPEATED SENTENCE DETECTION
# -----------------------------
print("\nTOP 30 REPEATED SENTENCES:")

if "full_text" in df.columns:
    all_sentences = []

    for text in df["full_text"].fillna("").astype(str):
        sentences = re.split(r'(?<=[\.\!\?])\s+', text)

        for s in sentences:
            s = s.strip()
            if len(s) >= 40:
                all_sentences.append(s)

    sent_counts = pd.Series(all_sentences).value_counts().head(30)

    for sent, count in sent_counts.items():
        print(f"{count:>4}  {sent[:250]}")

# -----------------------------
# TVNET WORD CONTEXT ANALYSIS
# -----------------------------
print("\nDETECTING 'TVNET' CONTEXTS")

pattern = r'\b\S*tvnet\S*\b'
contexts = []

for text in df["full_text"].fillna("").astype(str):
    matches = re.finditer(pattern, text, flags=re.IGNORECASE)

    for m in matches:
        start = max(m.start() - 40, 0)
        end = min(m.end() + 40, len(text))

        context = text[start:end]
        contexts.append(context.strip())

print("Total TVNET mentions:", len(contexts))

print("\nTOP 20 CONTEXTS AROUND 'TVNET':")

context_counts = Counter(contexts)

for context, count in context_counts.most_common(20):
    print(f"{count:>3}  {context}")

# -----------------------------
# TVNET PHRASE DETECTION
# -----------------------------
print("\nTOP PHRASES CONTAINING 'TVNET'")

phrases = []

for text in df["full_text"].fillna("").astype(str):
    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30}tvnet[A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30})',
        text,
        flags=re.IGNORECASE
    )

    for m in matches:
        phrases.append(m.strip())

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(20):
    print(f"{count:>3}  {phrase}")

ROWS: 178
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

FIRST 5 TITLES:
1. Pirmdiena bija karstākā diena pasaulē kopš mērījumu sākšanas
2. Ja Zeme attālinās no Saules, tad kāpēc ir tik karsts?
3. Video: Eiropu sasper tūkstošiem zibens šautru
4. Pagājusī vasara nogalināja 60 tūkstošus eiropiešu
5. Ģeologi: 1950. gadā sākās Cilvēka laikmets

LAST 5 TITLES:
1. Marsa izpētes misijā veiksmīgi palaista raķete
2. NASA zaudē kontaktus ar Marsa atmosfēras pētīšanas kosmosa kuģi (2)
3. Astronauta veselības problēmu dēļ NASA misija priekšlaicīgi atgriezusies no SKS (1)
4. NASA uz martu atliek astronautu lidojumu apkārt Mēnesim
5. VIDEO ⟩ Uz Rīgas apvedceļa notikusi traģiska avārija (32)

TEXT LENGTH SUMMARY:
Min: 91
Median: 1267
Max: 15430

TOP 30 REPEATED SENTENCES:
   5  Raksts tapis sadarbībā ar "LMT Viedtelevīziju".
   5  Ir atjaunota ceļu satiksmes negadījuma dēļ bloķētā satiksme uz Rīgas apvedceļa (Baltezer

# 1.2. data cleaning

In [ ]:
%%writefile clean_tvnet_reliable.py
from __future__ import annotations

import re
import html
import pandas as pd

IN_FILE = "tvnet_reliable_raw.csv"
OUT_FILE = "tvnet_reliable_clean.csv"

MIN_TEXT_LEN = 300

TOPIC_MAP = {
    "TVNET section 4211": "Ceļojumi",
    "TVNET section 4230": "Finanšu ziņas",
    "TVNET section 4237": "Klimats",
    "TVNET section 4282": "Atklājumi",
    "TVNET section 4292": "Kosmoss",
}

# --------------------------------------------------
# Title cleanup:
# 1) remove trailing comment counts like:
#    "... kuģi (2)" -> "... kuģi"
# 2) remove leading prefixes before and including "⟩"
#    "VIDEO ⟩ Virsraksts" -> "Virsraksts"
# --------------------------------------------------
TITLE_TRAILING_COUNT_PATTERN = re.compile(r"\s*\(\d+\)\s*$")
TITLE_PREFIX_BEFORE_ARROW_PATTERN = re.compile(r"^\s*[^⟩]{1,80}\s*⟩\s*")

# --------------------------------------------------
# Repeated / boilerplate text to remove
# --------------------------------------------------
REMOVE_PATTERNS = [
    r'Raksts\s*tapis\s*sadarbībā\s*ar\s*[\"“”„\']LMT\s*Viedtelevīziju[\"“”„\']\.?',
]

# --------------------------------------------------
# Source / branding phrases to remove fully
# --------------------------------------------------
SOURCE_PATTERNS = [
    r'[\"“”„\']?\s*TVNET\s*Bizness\s*[\"“”„\']?',
    r'[\"“”„\']?\s*TVNET\s*[\"“”„\']?',
]

COMPILED_REMOVE_PATTERNS = [
    re.compile(pat, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
    for pat in REMOVE_PATTERNS + SOURCE_PATTERNS
]


def normalize_text(text: object) -> str:
    """Basic technical cleanup: HTML, spaces, quotes, dashes."""
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize quotes/dashes
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00a0", " ").replace("\u200b", "")

    # Normalize spaces/newlines
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def clean_title(title: object) -> str:
    """Clean TVNET titles."""
    title = normalize_text(title)

    # Remove prefix like "VIDEO ⟩ "
    title = TITLE_PREFIX_BEFORE_ARROW_PATTERN.sub("", title)

    # Remove trailing "(number)"
    title = TITLE_TRAILING_COUNT_PATTERN.sub("", title)

    title = re.sub(r"\s+", " ", title).strip()
    return title


def clean_full_text(text: object) -> str:
    """Remove repeated boilerplate and source/branding phrases."""
    text = normalize_text(text)

    for pattern in COMPILED_REMOVE_PATTERNS:
        text = pattern.sub(" ", text)

    # Clean punctuation leftovers
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?]){2,}", r"\1", text)
    text = re.sub(r"\s*-\s*-\s*", " - ", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)
    text = re.sub(r"\s+", " ", text)

    # Strip awkward leading punctuation after removals
    text = re.sub(r"^[,.;:\-\s]+", "", text)

    return text.strip()


# --------------------------------------------------
# Load
# --------------------------------------------------
df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)
columns_before = df.columns.tolist()

# --------------------------------------------------
# Remove all articles whose titles start with FAKTOMĀTS ⟩
# --------------------------------------------------
if "title" in df.columns:
    title_series = df["title"].fillna("").astype(str)
    df = df[~title_series.str.contains(r"^\s*FAKTOMĀTS\s*⟩", case=False, regex=True)].copy()

rows_after_faktomats_filter = len(df)

# --------------------------------------------------
# Create cleaned columns
# --------------------------------------------------
df["title_clean"] = df["title"].apply(clean_title)
df["full_text_clean"] = df["full_text"].apply(clean_full_text)
df["text_length_clean"] = df["full_text_clean"].fillna("").astype(str).str.len()

# --------------------------------------------------
# Rename topics
# --------------------------------------------------
if "topic" in df.columns:
    df["topic"] = df["topic"].fillna("").astype(str).replace(TOPIC_MAP)

# --------------------------------------------------
# Remove empty / too short texts
# --------------------------------------------------
df = df[df["full_text_clean"].fillna("").astype(str).str.strip().str.len() >= MIN_TEXT_LEN].copy()

rows_after = len(df)
removed_rows_total = rows_before - rows_after
removed_faktomats_rows = rows_before - rows_after_faktomats_filter
removed_short_rows = rows_after_faktomats_filter - rows_after

# Reset index
df = df.reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# --------------------------------------------------
# Report
# --------------------------------------------------
columns_after = df.columns.tolist()
new_columns = [c for c in columns_after if c not in columns_before]

print("Saved:", OUT_FILE)
print("Rows before:", rows_before)
print("Rows after Faktomāts title filter:", rows_after_faktomats_filter)
print("Rows after final cleaning:", rows_after)
print("Removed Faktomāts-title rows:", removed_faktomats_rows)
print("Removed short/empty rows:", removed_short_rows)
print("Total removed rows:", removed_rows_total)

print("\nColumns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

print("\nNew columns added:")
print(new_columns if new_columns else "None")

Writing clean_tvnet_reliable.py


In [ ]:
!python clean_tvnet_reliable.py

Saved: tvnet_reliable_clean.csv
Rows before: 178
Rows after Faktomāts title filter: 175
Rows after final cleaning: 160
Removed Faktomāts-title rows: 3
Removed short/empty rows: 15
Total removed rows: 18

Columns before:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

Columns after:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

New columns added:
['title_clean', 'full_text_clean', 'text_length_clean']


# 1.3. dedublication

In [ ]:
%%writefile dedup_tvnet_reliable.py
from __future__ import annotations

import pandas as pd

IN_FILE = "tvnet_reliable_clean.csv"
OUT_FILE = "tvnet_reliable_dedup.csv"

df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Deduplicate by URL first
# ----------------------------------------
if "url" in df.columns:
    df["url"] = df["url"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["url"], keep="first")

rows_after_url = len(df)

# ----------------------------------------
# Deduplicate by cleaned text
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["full_text_clean"] = df["full_text_clean"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["full_text_clean"], keep="first")

rows_after_text = len(df)

# ----------------------------------------
# Recalculate clean text length
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["text_length_clean"] = df["full_text_clean"].str.len()

df = df.reset_index(drop=True)
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)
print("Rows before dedup:", rows_before)
print("Rows after URL dedup:", rows_after_url)
print("Rows after text dedup:", rows_after_text)
print("Total removed:", rows_before - len(df))

Writing dedup_tvnet_reliable.py


In [ ]:
!python dedup_tvnet_reliable.py

Saved: tvnet_reliable_dedup.csv
Rows before dedup: 160
Rows after URL dedup: 156
Rows after text dedup: 156
Total removed: 4


# 1.4. audit

In [ ]:
%%writefile audit_tvnet_reliable.py
from __future__ import annotations

import pandas as pd
import re
from collections import Counter

FILE = "tvnet_reliable_dedup.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) Top repeated sentences
# --------------------------------------------------
print("\nTOP 30 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text_clean"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(30):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 2) Top contexts containing TVNET
# Finds: TVNET, "TVNET", TVNET Bizness, etc.
# --------------------------------------------------
print("\nTOP 30 CONTEXTS CONTAINING 'TVNET':")

tvnet_pattern = r'\b\S*tvnet\S*\b'
contexts = []

for text in df["full_text_clean"].fillna("").astype(str):
    for match in re.finditer(tvnet_pattern, text, flags=re.IGNORECASE):
        start = max(match.start() - 40, 0)
        end = min(match.end() + 40, len(text))
        context = text[start:end].strip()
        contexts.append(context)

context_counts = Counter(contexts)

for context, count in context_counts.most_common(30):
    print(f"{count:>4}  {context}")

# --------------------------------------------------
# 3) Top phrases containing TVNET
# --------------------------------------------------
print("\nTOP 30 PHRASES CONTAINING 'TVNET':")

phrases = []

for text in df["full_text_clean"].fillna("").astype(str):
    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30}tvnet[A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30})',
        text,
        flags=re.IGNORECASE
    )

    for match in matches:
        phrase = re.sub(r"\s+", " ", match).strip()
        if phrase:
            phrases.append(phrase)

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(30):
    print(f"{count:>4}  {phrase}")

# --------------------------------------------------
# 4) Count rows containing TVNET
# --------------------------------------------------
print("\nTVNET MENTION COUNTS:")

title_hits = df["title_clean"].fillna("").astype(str).str.contains(
    tvnet_pattern, flags=re.IGNORECASE, regex=True
).sum()

text_hits = df["full_text_clean"].fillna("").astype(str).str.contains(
    tvnet_pattern, flags=re.IGNORECASE, regex=True
).sum()

print("TVNET in title_clean:", int(title_hits))
print("TVNET in full_text_clean:", int(text_hits))

Writing audit_tvnet_reliable.py


In [ ]:
!python audit_tvnet_reliable.py

ROWS: 156
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

TOP 30 REPEATED SENTENCES:
   3  gada vasaras sezonā RIX Rīgas lidosta piedāvā savienojumus ar vairāk nekā 100 tiešo regulāro un čartera lidojumu galamērķiem.
   3  Regulārus lidojumus no Rīgas nodrošina aviokompānijas airBaltic, AEGEAN, British Airways, Finnair, LOT Polish Airlines, Lufthansa, Norwegian, Ryanair, Transavia, Turkish Airlines, Uzbekistan Airways un Wizz Air.
   3  Vasaras sezonas RIX galamērķu karte apskatāma šeit.
   2  Paustie viedokļi un uzskati atspoguļo autora(-u) personīgos uzskatus un ne vienmēr sakrīt ar Eiropas Savienības vai Eiropas Komisijas Reģionālās politikas un pilsētpolitikas ģenerāldirektorāta viedokli.
   2  Ne Eiropas Savienība, ne finansējuma sniedzējs nav atbildīgs par paustajiem uzskatiem.
   2  Kā ziņots, pirmdien šī gada Nobela prēmija medicīnā tika pieš

In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "lv,en;q=0.9",
}
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

REQUEST_SLEEP = 0.7
BASE_LIST_URL = "https://www.lsm.lv/zinas/arzemes/"
PAGES = 14

TOPIC = "LSM | Ārzemes"
SOURCE = "LSM"
CLASS_LABEL = 0  # reliable

OUT_RAW = "lsm_arzemes_raw.csv"

# Required column order (keep identical across all datasets)
COLUMNS = [
    "title",
    "full_text",
    "source",
    "publication_date",
    "topic",
    "class_label",
    "url",
    "full_text_original",
    "text_length",
]

def normalize_url(u: str) -> str:
    parts = urlsplit(u)
    return urlunsplit((parts.scheme, parts.netloc, parts.path.rstrip("/"), "", ""))

def discover_urls(page_url: str) -> list[str]:
    r = SESSION.get(page_url, timeout=25)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    urls = set()
    for a in soup.find_all("a", href=True):
        full = urljoin(page_url, a["href"])

        # only LSM domain
        if not full.startswith("https://www.lsm.lv/"):
            continue

        # LSM articles typically contain /raksts/
        if "/raksts/" not in full:
            continue

        # keep only Ārzemes list-related pages (still allow /raksts/ articles)
        # (if you want stricter filtering, uncomment the next 2 lines)
        # if "/zinas/arzemes/" not in full:
        #     continue

        urls.add(normalize_url(full))

    return sorted(urls)

def extract_publication_date(soup: BeautifulSoup) -> str:
    # Try OpenGraph / meta first
    meta_time = soup.find("meta", attrs={"property": "article:published_time"})
    if meta_time and meta_time.get("content"):
        return meta_time["content"][:10]

    # Fallback: <time datetime="...">
    t = soup.find("time")
    if t and t.get("datetime"):
        return t["datetime"][:10]

    return ""

def extract_full_text(soup: BeautifulSoup) -> str:
    text_parts = []

    article = soup.find("article")
    if article:
        for p in article.find_all("p"):
            s = p.get_text(" ", strip=True)
            if s:
                text_parts.append(s)

    # Fallback: look for content-like containers if article text is too short
    if len(" ".join(text_parts)) < 50:
        for div in soup.find_all("div"):
            cls = " ".join(div.get("class", []))
            if any(k in cls.lower() for k in ["article", "content", "body", "text"]):
                for p in div.find_all("p"):
                    s = p.get_text(" ", strip=True)
                    if s:
                        text_parts.append(s)

    return "\n".join(text_parts).strip()

def scrape_article(article_url: str) -> dict | None:
    r = SESSION.get(article_url, timeout=25)
    if r.status_code != 200:
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    # title
    h1 = soup.find("h1")
    title = h1.get_text(" ", strip=True) if h1 else ""

    pub_date = extract_publication_date(soup)
    full_text = extract_full_text(soup)

    # Build record in the same schema as your other datasets
    full_text_original = full_text
    text_length = len(full_text) if isinstance(full_text, str) else 0

    return {
        "title": title,
        "full_text": full_text,
        "source": SOURCE,
        "publication_date": pub_date,
        "topic": TOPIC,
        "class_label": CLASS_LABEL,
        "url": normalize_url(article_url),
        "full_text_original": full_text_original,
        "text_length": text_length,
    }

# -----------------------
# MAIN
# -----------------------
all_urls = []
seen = set()

for p in range(1, PAGES + 1):
    page_url = BASE_LIST_URL if p == 1 else f"{BASE_LIST_URL}?p={p}"
    try:
        urls = discover_urls(page_url)
    except Exception as e:
        print(f"Page {p}/{PAGES}: ERROR {e}")
        continue

    added = 0
    for u in urls:
        if u not in seen:
            seen.add(u)
            all_urls.append(u)
            added += 1

    print(f"Page {p:02d}/{PAGES}: found {len(urls)}; added {added}; total_unique {len(all_urls)}")
    time.sleep(REQUEST_SLEEP)

print(f"\nTotal discovered LSM Ārzemes article URLs: {len(all_urls)}")

records = []
for i, u in enumerate(all_urls, 1):
    if i % 25 == 0 or i == 1:
        print(f"Scraping {i}/{len(all_urls)}...")
    try:
        rec = scrape_article(u)
        if rec:
            records.append(rec)
    except Exception:
        pass
    time.sleep(REQUEST_SLEEP)

df = pd.DataFrame(records)

# Ensure required columns exist and order is identical
for col in COLUMNS:
    if col not in df.columns:
        df[col] = ""

df = df[COLUMNS]

# Drop duplicate URLs (safety)
if len(df) > 0:
    df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

df.to_csv(OUT_RAW, index=False, encoding="utf-8")

print("\nSaved:", OUT_RAW)
print("Rows:", len(df))
print("Empty full_text:", int((df["full_text"].fillna("").str.len() == 0).sum()))
print("Min/Median/Max text length:",
      int(df["text_length"].min()) if len(df) else "n/a",
      int(df["text_length"].median()) if len(df) else "n/a",
      int(df["text_length"].max()) if len(df) else "n/a")

Page 01/14: found 94; added 94; total_unique 94
Page 02/14: found 96; added 22; total_unique 116
Page 03/14: found 96; added 20; total_unique 136
Page 04/14: found 96; added 20; total_unique 156
Page 05/14: found 96; added 20; total_unique 176
Page 06/14: found 97; added 21; total_unique 197
Page 07/14: found 96; added 19; total_unique 216
Page 08/14: found 96; added 20; total_unique 236
Page 09/14: found 96; added 20; total_unique 256
Page 10/14: found 96; added 20; total_unique 276
Page 11/14: found 96; added 20; total_unique 296
Page 12/14: found 97; added 21; total_unique 317
Page 13/14: found 96; added 20; total_unique 337
Page 14/14: found 96; added 20; total_unique 357

Total discovered LSM Ārzemes article URLs: 357
Scraping 1/357...
Scraping 25/357...
Scraping 50/357...
Scraping 75/357...
Scraping 100/357...
Scraping 125/357...
Scraping 150/357...
Scraping 175/357...
Scraping 200/357...
Scraping 225/357...
Scraping 250/357...
Scraping 275/357...
Scraping 300/357...
Scraping 325

2.# New Section

# 2.LSM

# 2.1. raw data inspection

In [ ]:
import pandas as pd
import re
from collections import Counter

FILE = "lsm_arzemes_raw.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# -----------------------------
# TITLE INSPECTION
# -----------------------------
print("\nFIRST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).head(5), start=1):
        print(f"{i}. {title}")

print("\nLAST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).tail(5), start=1):
        print(f"{i}. {title}")

# -----------------------------
# TEXT LENGTH STATS
# -----------------------------
print("\nTEXT LENGTH SUMMARY:")
if "full_text" in df.columns:
    lengths = df["full_text"].fillna("").astype(str).str.len()
    print("Min:", int(lengths.min()))
    print("Median:", int(lengths.median()))
    print("Max:", int(lengths.max()))

# -----------------------------
# REPEATED SENTENCE DETECTION
# -----------------------------
print("\nTOP 30 REPEATED SENTENCES:")

if "full_text" in df.columns:
    all_sentences = []

    for text in df["full_text"].fillna("").astype(str):
        sentences = re.split(r'(?<=[\.\!\?])\s+', text)

        for s in sentences:
            s = s.strip()
            if len(s) >= 40:
                all_sentences.append(s)

    sent_counts = pd.Series(all_sentences).value_counts().head(30)

    for sent, count in sent_counts.items():
        print(f"{count:>4}  {sent[:250]}")

# -----------------------------
# LSM WORD CONTEXT ANALYSIS
# -----------------------------
print("\nDETECTING 'LSM' CONTEXTS")

pattern = r'\b\S*lsm\S*\b'
contexts = []

for text in df["full_text"].fillna("").astype(str):
    matches = re.finditer(pattern, text, flags=re.IGNORECASE)

    for m in matches:
        start = max(m.start() - 40, 0)
        end = min(m.end() + 40, len(text))

        context = text[start:end]
        contexts.append(context.strip())

print("Total LSM mentions:", len(contexts))

print("\nTOP 20 CONTEXTS AROUND 'LSM':")

context_counts = Counter(contexts)

for context, count in context_counts.most_common(20):
    print(f"{count:>3}  {context}")

# -----------------------------
# LSM PHRASE DETECTION
# -----------------------------
print("\nTOP PHRASES CONTAINING 'LSM'")

phrases = []

for text in df["full_text"].fillna("").astype(str):
    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30}lsm[A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30})',
        text,
        flags=re.IGNORECASE
    )

    for m in matches:
        phrases.append(m.strip())

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(20):
    print(f"{count:>3}  {phrase}")

ROWS: 357
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

FIRST 5 TITLES:
1. Alesja un Darja no Mariupoles nebaidās mācīties jauno
2. Esam gatavi runāt par sarežģītām vēstures tēmām. Raidījums «Izvēle Nr. 1» spējis izlauzties cauri izklaides troksnim
3. Oleksijs Zaričanskis: Kādēļ un kas vērš naidu pret bēgļiem internetā
4. Jūlija Kaļada: Kur vieglāk būt sievietei – Latvijā vai Baltkrievijā? Bēgles domas
5. Oksana Kiriļuka no Kijivas apgabala zina, kā veidot sadarbību starp abu valstu pašvaldībām

LAST 5 TITLES:
1. Medijs: Kremlim lojālie mediji gatavo Krievijas iedzīvotājus septembrī gaidāmajām Valsts domes vēlēšanām
2. krievijas iebrukums Ukrainā
3. Tramps liek publiskot visu informāciju par citplanētiešiem
4. Ukrainai izdevies novērst Krievijas specdienesta mēģinājumu nogalināt sabiedriski aktīvus ukraiņus
5. Venecuēlā pieņemtais amnestijas likums varētu  ļaut atgūt brīvību daudziem politieslodzītajie

# 2.2. data cleaning

In [ ]:
%%writefile clean_lsm_arzemes.py
from __future__ import annotations

import re
import html
import pandas as pd

IN_FILE = "lsm_arzemes_raw.csv"
OUT_FILE = "lsm_arzemes_clean.csv"

MIN_TEXT_LEN = 300

# --------------------------------------------------
# Title cleanup:
# 1) remove leading speaker/source prefixes before colon
#    e.g. "Medijs: ..." -> "..."
#         "Oleksijs Zaričanskis: ..." -> "..."
# 2) remove trailing comment counts like "(11)" at title end
# --------------------------------------------------
TITLE_TRAILING_COUNT_PATTERN = re.compile(r"\s*\(\d+\)\s*$")
TITLE_PREFIX_COLON_PATTERN = re.compile(
    r"^\s*[A-ZĀČĒĢĪĶĻŅŠŪŽa-zāčēģīķļņšūž0-9][A-ZĀČĒĢĪĶĻŅŠŪŽa-zāčēģīķļņšūž0-9\s\.\-]{0,60}:\s*"
)

# --------------------------------------------------
# Remove repeated / boilerplate sentences and phrases
# --------------------------------------------------
REMOVE_PATTERNS = [
    r"Par\s+faktu\s+kļūdām\s+lūdzam\s+ziņot\s+e-?pastā\s+\[email protected\]\s*\.?",
    r"Iezīmējiet\s+tekstu\s+un\s+spiediet\s+uz\s+Ziņot\s+par\s+kļūdu\s+pogas,\s+lai\s+nosūtītu\s+labojamo\s+teksta\s+fragmentu\s+redaktoram!?",
    r"Iezīmējiet\s+tekstu\s+un\s+spiediet\s+Ctrl\+Enter\s*,?\s+lai\s+nosūtītu\s+labojamo\s+teksta\s+fragmentu\s+redaktoram!?",
    r"Sūti\s+ziņu\s+LSM\.lv\s+redakcijai!?",
    r"mēs\s+tos\s+publicēsim\s+LSM\.lv",
]

# --------------------------------------------------
# Source / branding phrases to remove
# --------------------------------------------------
SOURCE_PATTERNS = [
    r"\bPortāls\s+LSM\.lv\b",
    r"\bportāla\s+LSM\.lv\b",
    r"\bPortālā\s+LSM\.lv\b",
    r"\bLSM\.lv\b",
    r"\bLSM\b",
    r"Sūti\s+ziņu\s+LSM\.lv\s+redakcijai!?",
]

# Compile regex patterns
COMPILED_REMOVE_PATTERNS = [
    re.compile(pat, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
    for pat in REMOVE_PATTERNS + SOURCE_PATTERNS
]


def normalize_text(text: object) -> str:
    """Basic technical cleanup: HTML, spaces, quotes, dashes."""
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize quotes / dashes / spaces
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00a0", " ").replace("\u200b", "")

    # Normalize whitespace
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def clean_title(title: object) -> str:
    """
    Clean title:
    - remove leading prefix before colon
    - remove trailing (number)
    """
    title = normalize_text(title)

    title = TITLE_PREFIX_COLON_PATTERN.sub("", title)
    title = TITLE_TRAILING_COUNT_PATTERN.sub("", title)

    title = re.sub(r"\s+", " ", title).strip()
    return title


def clean_full_text(text: object) -> str:
    """
    Clean full text:
    - remove leading labels like 'KONTEKSTS:', 'ĪSUMĀ:', 'Foto:'
    - remove repeated boilerplate
    - remove branding/source phrases
    """
    text = normalize_text(text)

    # Remove leading labels only at the beginning
    text = re.sub(r"^\s*KONTEKSTS:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*ĪSUMĀ:\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*Foto:\s*", "", text, flags=re.IGNORECASE)

    for pattern in COMPILED_REMOVE_PATTERNS:
        text = pattern.sub(" ", text)

    # Clean punctuation leftovers
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?]){2,}", r"\1", text)
    text = re.sub(r"\s*-\s*-\s*", " - ", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)
    text = re.sub(r"\s+", " ", text)

    # Strip awkward leading punctuation after removals
    text = re.sub(r"^[,.;:\-\s]+", "", text)

    return text.strip()


# --------------------------------------------------
# Load
# --------------------------------------------------
df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)
columns_before = df.columns.tolist()

# --------------------------------------------------
# Create cleaned columns
# --------------------------------------------------
df["title_clean"] = df["title"].apply(clean_title)
df["full_text_clean"] = df["full_text"].apply(clean_full_text)
df["text_length_clean"] = df["full_text_clean"].fillna("").astype(str).str.len()

# --------------------------------------------------
# Remove empty / too short texts
# --------------------------------------------------
df = df[df["full_text_clean"].fillna("").astype(str).str.strip().str.len() >= MIN_TEXT_LEN].copy()

rows_after = len(df)
removed_rows = rows_before - rows_after

# Reset index
df = df.reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# --------------------------------------------------
# Report
# --------------------------------------------------
columns_after = df.columns.tolist()
new_columns = [c for c in columns_after if c not in columns_before]

print("Saved:", OUT_FILE)
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed rows:", removed_rows)

print("\nColumns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

print("\nNew columns added:")
print(new_columns if new_columns else "None")

Writing clean_lsm_arzemes.py


In [ ]:
!python clean_lsm_arzemes.py

Saved: lsm_arzemes_clean.csv
Rows before: 357
Rows after: 353
Removed rows: 4

Columns before:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

Columns after:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

New columns added:
['title_clean', 'full_text_clean', 'text_length_clean']


#  2.3. dedublication

In [ ]:
%%writefile dedup_lsm_arzemes.py
from __future__ import annotations

import pandas as pd

IN_FILE = "lsm_arzemes_clean.csv"
OUT_FILE = "lsm_arzemes_dedup.csv"

df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Deduplicate by URL
# ----------------------------------------
if "url" in df.columns:
    df["url"] = df["url"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["url"], keep="first")

rows_after_url = len(df)

# ----------------------------------------
# Deduplicate by cleaned text
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["full_text_clean"] = df["full_text_clean"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["full_text_clean"], keep="first")

rows_after_text = len(df)

# ----------------------------------------
# Recalculate clean text length
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["text_length_clean"] = df["full_text_clean"].str.len()

df = df.reset_index(drop=True)

df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)
print("Rows before dedup:", rows_before)
print("Rows after URL dedup:", rows_after_url)
print("Rows after text dedup:", rows_after_text)
print("Total removed:", rows_before - len(df))

Writing dedup_lsm_arzemes.py


In [ ]:
!python dedup_lsm_arzemes.py

Saved: lsm_arzemes_dedup.csv
Rows before dedup: 353
Rows after URL dedup: 353
Rows after text dedup: 348
Total removed: 5


# 2.4. audit

In [ ]:
%%writefile audit_lsm_arzemes.py
from __future__ import annotations

import pandas as pd
import re
from collections import Counter

FILE = "lsm_arzemes_dedup.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) Top repeated sentences
# --------------------------------------------------
print("\nTOP 30 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text_clean"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(30):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 2) Detect real LSM mentions only
# Finds only:
# - LSM
# - LSM.lv
# --------------------------------------------------
print("\nTOP 30 CONTEXTS CONTAINING 'LSM':")

lsm_pattern = r'\bLSM(?:\.lv)?\b'
contexts = []

for text in df["full_text_clean"].fillna("").astype(str):
    for match in re.finditer(lsm_pattern, text, flags=re.IGNORECASE):
        start = max(match.start() - 40, 0)
        end = min(match.end() + 40, len(text))
        context = text[start:end].strip()
        contexts.append(context)

context_counts = Counter(contexts)

for context, count in context_counts.most_common(30):
    print(f"{count:>4}  {context}")

# --------------------------------------------------
# 3) Top phrases containing real LSM mentions
# --------------------------------------------------
print("\nTOP 30 PHRASES CONTAINING 'LSM':")

phrases = []

for text in df["full_text_clean"].fillna("").astype(str):
    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30}\bLSM(?:\.lv)?\b[A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30})',
        text,
        flags=re.IGNORECASE
    )

    for match in matches:
        phrase = re.sub(r"\s+", " ", match).strip()
        if phrase:
            phrases.append(phrase)

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(30):
    print(f"{count:>4}  {phrase}")

# --------------------------------------------------
# 4) Count rows containing real LSM mentions
# --------------------------------------------------
print("\nLSM MENTION COUNTS:")

title_hits = df["title_clean"].fillna("").astype(str).str.contains(
    lsm_pattern, flags=re.IGNORECASE, regex=True
).sum()

text_hits = df["full_text_clean"].fillna("").astype(str).str.contains(
    lsm_pattern, flags=re.IGNORECASE, regex=True
).sum()

print("LSM in title_clean:", int(title_hits))
print("LSM in full_text_clean:", int(text_hits))

Overwriting audit_lsm_arzemes.py


In [ ]:
!python audit_lsm_arzemes.py

ROWS: 348
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

TOP 30 REPEATED SENTENCES:
  50  februārī ASV un Izraēla sāka jaunus triecienus pret Irānu.
  49  ASV un Izraēlas līderi paziņoja, ka triecienu mērķis ir panākt režīma maiņu Irānā.
  48  gada jūnijā Izraēla un ASV veica triecienus pret Irānas kodolprogrammas objektiem.
  44  gadā Irānā pie varas nāca islāma fundamentālistu režīms, kas sarāva jebkādas saites ar Izraēlu.
  44  Irāna uzskata, ka Izraēlas valsts nedrīkst eksistēt; Teherānas režīms draudējis noslaucīt Izraēlu no pasaules kartes.
  44  Irāna tiek uzskatīta par bīstamāko Izraēlas ienaidnieci reģionā, taču Izraēlas rīcībā ir kodolieroči un pasaules spēcīgākās lielvaras ASV atbalsts.
  44  Irānas režīms finansē tādas teroristu organizācijas kā "Hamās" un "Hizbullāh", kas cīnās pret Izraēlu.
  44  oktobrī, kad "Hamās" teroristi sarīkoja

# **3. LTV**

In [ ]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlsplit, urlunsplit

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "lv,en;q=0.9",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

REQUEST_SLEEP = 0.6

SOURCE = "LTV"
TOPIC_DEFAULT = "Aktuāli"
CLASS_LABEL = 0  # reliable class

OUT_FILE = "ltv_reliable_raw.csv"

URLS_RAW = """
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskais-medijs-neatspogulos-krievijas-un-baltkrievijas-sportistu-dalibu-ziemas-olimpiskajas-speles.id370650?page=1,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskais-medijs-piedavas-plasu-2026-gada-ziemas-olimpisko-spelu-atspogulojumu.id370637?page=1,
https://ltv.lsm.lv/aktuali/atklats-latvijas-sabiedriska-medija-gada-balvas-kultura-kilograms-kulturas-2025-finala-balsojums.id370574?page=1,
https://ltv.lsm.lv/aktuali/jau-sestdien-latvijas-sabiedriska-medija-konkursa-supernova-pirmaja-pusfinala-klus-zinami-pirmie-pieci-finalisti.id370480?page=1,
https://ltv.lsm.lv/aktuali/latvijas-televizijas-muzikali-izklaidejosa-sova-pardziedi-mani-otras-sezonas-uzvaretaja-ir-emilija.id370081?page=1,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriska-medija-konkursa-latvijas-jaunais-virtuozs-noskaidroti-sesi-finalisti.id369704?page=1,
https://ltv.lsm.lv/aktuali/sacies-kilograms-kulturas-2025-ziemas-balsojums.id369630?page=1,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-tapusi-jauna-vesturiska-dokumentala-filma-baltijas-babele.id369572?page=2,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-tapis-jauns-raidijums-izvele-nr-1.id369351?page=2,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-tapusi-dokumentala-filma-maestro-raimonds-pauls-dziesmina-par-prieku.id369298?page=2,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskais-medijs-nosledz-pirmo-darbibas-gadu-un-uzsak-strategijas-ieviesanu.id369061?page=2,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskais-medijs-aicina-sagaidit-jauno-gadu-kopa-ar-pardziedi-mani-vecgada-sovu.id368996?page=2,
https://ltv.lsm.lv/aktuali/ir-noskaidrots-latvijas-sabiedriska-medija-dziesmu-konkursa-supernova-pusfinalu-dziesmu-sadalijums.id368869?page=2,
https://ltv.lsm.lv/aktuali/svetku-laiks-latvijas-televizija-filmas-koncerti-un-ipasi-notikumi-visai-gimenei.id368866?page=2,
https://ltv.lsm.lv/aktuali/labdaribas-maratona-dod-pieci-saziedota-rekordliela-summa-1-001-388-eiro.id368443?page=3,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-tapusi-dokumentala-filma-antra-liedskalnina-kaislibu-vara.id367951?page=3,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskais-medijs-atjauno-latvijas-dalibu-starptautiskaja-konkursa-eirovizijas-jaunie-muziki.id367885?page=3,
https://ltv.lsm.lv/aktuali/izsludinats-latvijas-sabiedriska-medija-dokumentalo-isfilmu-konkurss-cikla-latvijas-kods-latvija-sodien.id367422?page=3,
https://ltv.lsm.lv/aktuali/sakas-dziesmu-apmaina-pret-ziedojumiem-labdaribas-maratonam-dod-pieci.id367387?page=3,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-basketbola-tiesraides-ltv7-un-replaylv-basketbola-studija-ar-osenieku-peineru-un-gulbi.id367068?page=3,
https://ltv.lsm.lv/aktuali/sogad-dod-pieci-stikla-studija-tiks-ieslegti-tris-didzeji.id366908?page=4,
https://ltv.lsm.lv/aktuali/izveleti-latvijas-sabiedriska-medija-dziesmu-konkursa-supernova-pusfinalisti.id366708?page=4,
https://ltv.lsm.lv/aktuali/latvijas-republikas-neatkaribas-pasludinasanas-diena-latvijas-sabiedriskais-medijs-piedava-ipasu-svetku-programmu.id366429?page=4,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-tapusi-dokumentala-filma-par-vienu-no-latviesu-hiphopa-pamatlicejiem-tas-pats-es-gustavo.id366143?page=4,
https://ltv.lsm.lv/aktuali/latvijas-televizija-no-novembra-parraidis-nozimigakos-fiba-basketbola-turnirus-tsk-latvijas-nacionalo-basketbola-izlasu-speles.id365830?page=4,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-ar-otro-sezonu-atgriezas-latvijas-televizijas-muzikali-izklaidejoss-sovs-pardziedi-mani.id365374?page=5,
https://ltv.lsm.lv/aktuali/latvijas-un-starptautiskie-eksperti-diskute-par-sabiedriska-medija-attistibu-un-sabiedribas-uzticesanos.id364854?page=5,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriskaja-medija-jauns-senioru-iepazisanas-raidijums-sirdslietas.id364428?page=5,
https://ltv.lsm.lv/aktuali/evita-gutmane-no-vircavas-klust-par-raidijuma-istas-latvju-saimnieces-10-sezonas-uzvaretaju.id364281?page=5,
https://ltv.lsm.lv/aktuali/latvijas-sabiedriska-medija-un-ziedotlv-atbalsta-akcija-ukrainai-saziedoti-240-004-eiro.id362140?page=6
"""

def normalize_url_drop_query(u: str) -> str:
    u = u.strip().strip(",")
    if not u:
        return ""
    parts = urlsplit(u)
    return urlunsplit((parts.scheme, parts.netloc, parts.path.rstrip("/"), "", ""))

def extract_urls(text_blob: str) -> list[str]:
    raw = re.split(r"[\s,]+", text_blob.strip())
    urls = [x for x in raw if x.startswith("http")]
    seen = set()
    out = []
    for u in urls:
        nu = normalize_url_drop_query(u)
        if nu and nu not in seen:
            seen.add(nu)
            out.append(nu)
    return out

def scrape_ltv_article(url: str) -> dict:
    r = SESSION.get(url, timeout=25)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    h1 = soup.select_one("h1.ltv-article--title") or soup.find("h1")
    title = h1.get_text(" ", strip=True) if h1 else ""

    t = soup.select_one("time.ltv-article--time") or soup.find("time")
    publication_date = t.get("datetime", "")[:10] if t and t.get("datetime") else ""

    lead = ""
    lead_p = soup.select_one('div[aria-label="Raksta ievads"] p')
    if lead_p:
        lead = lead_p.get_text(" ", strip=True)

    body_parts = []
    body_div = (
        soup.select_one('div.rich-text.ltv-article--body[itemprop="articleBody"]')
        or soup.select_one("div.rich-text.ltv-article--body")
    )
    if body_div:
        for p in body_div.find_all("p"):
            s = p.get_text(" ", strip=True)
            if s:
                body_parts.append(s)

    full_text = "\n".join([x for x in [lead] if x] + body_parts).strip()

    return {
        "title": title,
        "full_text": full_text,
        "source": SOURCE,
        "publication_date": publication_date,
        "topic": TOPIC_DEFAULT,
        "class_label": CLASS_LABEL,
        "url": url,
    }

# -----------------------
# MAIN
# -----------------------
urls = extract_urls(URLS_RAW)
print("Total URLs:", len(urls))

rows = []
for i, u in enumerate(urls, 1):
    print(f"Scraping {i}/{len(urls)}")
    try:
        rows.append(scrape_ltv_article(u))
    except Exception as e:
        print("Failed:", u, "->", e)
    time.sleep(REQUEST_SLEEP)

df = pd.DataFrame(rows)

# Drop exact duplicate URLs
if len(df) > 0:
    df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

# Add required columns
df["full_text_original"] = df["full_text"]
df["text_length"] = df["full_text"].fillna("").astype(str).str.len()

# Force required column order (identical schema)
COL_ORDER = [
    "title",
    "full_text",
    "source",
    "publication_date",
    "topic",
    "class_label",
    "url",
    "full_text_original",
    "text_length",
]
df = df.reindex(columns=COL_ORDER)

df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("\nSaved:", OUT_FILE)
print("Rows:", len(df))
print("Empty full_text:", int(df["full_text"].fillna("").astype(str).str.len().eq(0).sum()))
print("Min/Median/Max text length:",
      (int(df["text_length"].min()), int(df["text_length"].median()), int(df["text_length"].max())) if len(df) > 0 else "n/a")

Total URLs: 30
Scraping 1/30
Scraping 2/30
Scraping 3/30
Scraping 4/30
Scraping 5/30
Scraping 6/30
Scraping 7/30
Scraping 8/30
Scraping 9/30
Scraping 10/30
Scraping 11/30
Scraping 12/30
Scraping 13/30
Scraping 14/30
Scraping 15/30
Scraping 16/30
Scraping 17/30
Scraping 18/30
Scraping 19/30
Scraping 20/30
Scraping 21/30
Scraping 22/30
Scraping 23/30
Scraping 24/30
Scraping 25/30
Scraping 26/30
Scraping 27/30
Scraping 28/30
Scraping 29/30
Scraping 30/30

Saved: ltv_reliable_raw.csv
Rows: 30
Empty full_text: 0
Min/Median/Max text length: (1215, 2859, 17889)


# 3.1. raw data inspection

In [ ]:
import pandas as pd
import re
from collections import Counter

FILE = "ltv_reliable_raw.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# -----------------------------
# TITLE INSPECTION
# -----------------------------
print("\nFIRST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).head(5), start=1):
        print(f"{i}. {title}")

print("\nLAST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).tail(5), start=1):
        print(f"{i}. {title}")

# -----------------------------
# TEXT LENGTH STATS
# -----------------------------
print("\nTEXT LENGTH SUMMARY:")
if "full_text" in df.columns:
    lengths = df["full_text"].fillna("").astype(str).str.len()
    print("Min:", int(lengths.min()))
    print("Median:", int(lengths.median()))
    print("Max:", int(lengths.max()))

# -----------------------------
# REPEATED SENTENCE DETECTION
# -----------------------------
print("\nTOP 30 REPEATED SENTENCES:")

if "full_text" in df.columns:
    all_sentences = []

    for text in df["full_text"].fillna("").astype(str):
        sentences = re.split(r'(?<=[\.\!\?])\s+', text)

        for s in sentences:
            s = s.strip()
            if len(s) >= 40:
                all_sentences.append(s)

    sent_counts = pd.Series(all_sentences).value_counts().head(30)

    for sent, count in sent_counts.items():
        print(f"{count:>4}  {sent[:250]}")

# -----------------------------
# LTV WORD CONTEXT ANALYSIS
# -----------------------------
print("\nDETECTING 'LTV' CONTEXTS")

pattern = r'\bLTV(?:\.lv)?\b'
contexts = []

for text in df["full_text"].fillna("").astype(str):
    matches = re.finditer(pattern, text, flags=re.IGNORECASE)

    for m in matches:
        start = max(m.start() - 40, 0)
        end = min(m.end() + 40, len(text))

        context = text[start:end]
        contexts.append(context.strip())

print("Total LTV mentions:", len(contexts))

print("\nTOP 20 CONTEXTS AROUND 'LTV':")

context_counts = Counter(contexts)

for context, count in context_counts.most_common(20):
    print(f"{count:>3}  {context}")

# -----------------------------
# LTV PHRASE DETECTION
# -----------------------------
print("\nTOP PHRASES CONTAINING 'LTV'")

phrases = []

for text in df["full_text"].fillna("").astype(str):
    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30}\bLTV(?:\.lv)?\b[A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30})',
        text,
        flags=re.IGNORECASE
    )

    for m in matches:
        phrases.append(re.sub(r"\s+", " ", m).strip())

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(20):
    print(f"{count:>3}  {phrase}")

ROWS: 30
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

FIRST 5 TITLES:
1. Latvijas Sabiedriskais medijs neatspoguļos Krievijas un Baltkrievijas sportistu dalību Ziemas olimpiskajās spēlēs
2. Latvijas Sabiedriskais medijs piedāvās plašu 2026. gada ziemas olimpisko spēļu atspoguļojumu
3. Atklāts Latvijas Sabiedriskā medija gada balvas kultūrā "Kilograms kultūras 2025" fināla balsojums
4. Jau sestdien Latvijas Sabiedriskā medija konkursa "Supernova" pirmajā pusfinālā kļūs zināmi pirmie pieci finālisti
5. Latvijas Televīzijas muzikāli izklaidējošā šova "Pārdziedi mani!" otrās sezonas uzvarētāja ir Emilija

LAST 5 TITLES:
1. Latvijas Sabiedriskajā medijā ar otro sezonu atgriežas Latvijas Televīzijas muzikāli izklaidējošs šovs "Pārdziedi mani!"
2. Latvijas un starptautiskie eksperti diskutē par sabiedriskā medija attīstību un sabiedrības uzticēšanos
3. Latvijas Sabiedriskajā medijā jauns senioru iepazīšanās 

# 3.2. data cleaning

In [ ]:
%%writefile clean_ltv_reliable.py
from __future__ import annotations

import re
import html
import pandas as pd

IN_FILE = "ltv_reliable_raw.csv"
OUT_FILE = "ltv_reliable_clean.csv"

MIN_TEXT_LEN = 300

# --------------------------------------------------
# Title cleanup:
# remove trailing comment counts like "(11)" if present
# --------------------------------------------------
TITLE_TRAILING_COUNT_PATTERN = re.compile(r"\s*\(\d+\)\s*$")

# --------------------------------------------------
# Source removal
# --------------------------------------------------
SOURCE_PATTERNS = [
    r"\(LTV\)",
    r"\bLTV\b",
]

COMPILED_SOURCE_PATTERNS = [
    re.compile(p, flags=re.IGNORECASE | re.UNICODE)
    for p in SOURCE_PATTERNS
]


def normalize_text(text: object) -> str:
    """Basic technical cleanup."""
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    # remove HTML
    text = re.sub(r"<[^>]+>", " ", text)

    # normalize punctuation
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00a0", " ").replace("\u200b", "")

    # normalize whitespace
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def clean_title(title: object) -> str:
    title = normalize_text(title)
    title = TITLE_TRAILING_COUNT_PATTERN.sub("", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def clean_full_text(text: object) -> str:
    text = normalize_text(text)

    for pattern in COMPILED_SOURCE_PATTERNS:
        text = pattern.sub(" ", text)

    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?]){2,}", r"\1", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# --------------------------------------------------
# Load
# --------------------------------------------------
df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)
columns_before = df.columns.tolist()

# --------------------------------------------------
# Create cleaned columns
# --------------------------------------------------
df["title_clean"] = df["title"].apply(clean_title)
df["full_text_clean"] = df["full_text"].apply(clean_full_text)
df["text_length_clean"] = df["full_text_clean"].fillna("").astype(str).str.len()

# --------------------------------------------------
# Remove short texts
# --------------------------------------------------
df = df[df["text_length_clean"] >= MIN_TEXT_LEN].copy()

rows_after = len(df)
removed_rows = rows_before - rows_after

df = df.reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# --------------------------------------------------
# Report
# --------------------------------------------------
columns_after = df.columns.tolist()
new_columns = [c for c in columns_after if c not in columns_before]

print("Saved:", OUT_FILE)
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed rows:", removed_rows)

print("\nColumns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

print("\nNew columns added:")
print(new_columns if new_columns else "None")

Writing clean_ltv_reliable.py


In [ ]:
!python clean_ltv_reliable.py

Saved: ltv_reliable_clean.csv
Rows before: 30
Rows after: 30
Removed rows: 0

Columns before:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

Columns after:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

New columns added:
['title_clean', 'full_text_clean', 'text_length_clean']


# 3.3. dedublication

In [ ]:
%%writefile dedup_ltv_reliable.py
from __future__ import annotations

import pandas as pd

IN_FILE = "ltv_reliable_clean.csv"
OUT_FILE = "ltv_reliable_dedup.csv"

df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Deduplicate by URL
# ----------------------------------------
if "url" in df.columns:
    df["url"] = df["url"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["url"], keep="first")

rows_after_url = len(df)

# ----------------------------------------
# Deduplicate by cleaned text
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["full_text_clean"] = df["full_text_clean"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["full_text_clean"], keep="first")

rows_after_text = len(df)

# ----------------------------------------
# Recalculate clean text length
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["text_length_clean"] = df["full_text_clean"].str.len()

df = df.reset_index(drop=True)
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)
print("Rows before dedup:", rows_before)
print("Rows after URL dedup:", rows_after_url)
print("Rows after text dedup:", rows_after_text)
print("Total removed:", rows_before - len(df))

Writing dedup_ltv_reliable.py


In [ ]:
!python dedup_ltv_reliable.py

Saved: ltv_reliable_dedup.csv
Rows before dedup: 30
Rows after URL dedup: 30
Rows after text dedup: 30
Total removed: 0


# 3.4. audit

In [ ]:
%%writefile audit_ltv_reliable.py
from __future__ import annotations

import pandas as pd
import re
from collections import Counter

FILE = "ltv_reliable_dedup.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) Top repeated sentences
# --------------------------------------------------
print("\nTOP 30 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text_clean"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(30):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 2) Detect real LTV mentions only
# Finds only:
# - LTV
# - LTV.lv
# --------------------------------------------------
print("\nTOP 30 CONTEXTS CONTAINING 'LTV':")

ltv_pattern = r'\bLTV(?:\.lv)?\b'
contexts = []

for text in df["full_text_clean"].fillna("").astype(str):
    for match in re.finditer(ltv_pattern, text, flags=re.IGNORECASE):
        start = max(match.start() - 40, 0)
        end = min(match.end() + 40, len(text))
        context = text[start:end].strip()
        contexts.append(context)

context_counts = Counter(contexts)

for context, count in context_counts.most_common(30):
    print(f"{count:>4}  {context}")

# --------------------------------------------------
# 3) Top phrases containing real LTV mentions
# --------------------------------------------------
print("\nTOP 30 PHRASES CONTAINING 'LTV':")

phrases = []

for text in df["full_text_clean"].fillna("").astype(str):
    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30}\bLTV(?:\.lv)?\b[A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30})',
        text,
        flags=re.IGNORECASE
    )

    for match in matches:
        phrase = re.sub(r"\s+", " ", match).strip()
        if phrase:
            phrases.append(phrase)

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(30):
    print(f"{count:>4}  {phrase}")

# --------------------------------------------------
# 4) Count rows containing real LTV mentions
# --------------------------------------------------
print("\nLTV MENTION COUNTS:")

title_hits = df["title_clean"].fillna("").astype(str).str.contains(
    ltv_pattern, flags=re.IGNORECASE, regex=True
).sum()

text_hits = df["full_text_clean"].fillna("").astype(str).str.contains(
    ltv_pattern, flags=re.IGNORECASE, regex=True
).sum()

print("LTV in title_clean:", int(title_hits))
print("LTV in full_text_clean:", int(text_hits))

Writing audit_ltv_reliable.py


In [ ]:
!python audit_ltv_reliable.py

ROWS: 30
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

TOP 30 REPEATED SENTENCES:
   3  Par konkursu "Supernova" "Supernova" ir LSM organizēts dziesmu konkurss, kura mērķis ir izvēlēties Latvijas pārstāvi Eirovīzijas dziesmu konkursam.
   3  "Supernova" pirmo reizi norisinājās 2015.
   3  gadā, aizstājot iepriekšējos nacionālās atlases formātus "Eirodziesma" (2000-2012) un "Dziesma" (2013-2014).
   3  "Dod pieci!" laikā saziedotie līdzekļi nonāks Ziedot.lv rīcībā un labdarības organizācija palīdzēs apmaksāt fizioterapijas, ergoterapijas, ārstnieciskās masāžas, ūdensdziedniecības un fizikālās procedūras, kas nav tiešā valsts vai pašvaldības atbildī
   3  Katrs ziedojums tādējādi kļūs par ieguldījumu bērna spējā kustēties, mācīties, kļūt patstāvīgiem un dzīvot pilnvērtīgi.
   3  Labdarības maratona virsmērķis ir cilvēciskāka vide - katrs "Dod pieci!"

# 4.DELFI

In [ ]:
code = r'''
import re
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "lv,en;q=0.9",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

REQUEST_SLEEP = 0.8
TARGET_PER_CATEGORY = 200

CATEGORIES = [
    {
        "topic": "Biznesa vide",
        "start_url": "https://www.delfi.lv/bizness/37264250/biznesa_vide",
        "source": "Delfi (Bizness)",
        "class_label": 0,
    },
    {
        "topic": "Eiropas ziņas",
        "start_url": "https://www.delfi.lv/bizness/eiropas-zinas",
        "source": "Delfi (Bizness)",
        "class_label": 0,
    },
    {
        "topic": "Pasaule",
        "start_url": "https://www.delfi.lv/bizness/37293424/pasaule",
        "source": "Delfi (Bizness)",
        "class_label": 0,
    },
]

def normalize_ws(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t\r\f\v]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

def build_page_url(start_url: str, page: int) -> str:
    # Delfi section pagination commonly uses ?page=N
    if page <= 1:
        return start_url
    return f"{start_url}?page={page}"

def extract_listing_urls(html: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")
    urls = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if not href:
            continue

        if href.startswith("/"):
            href = "https://www.delfi.lv" + href

        if not href.startswith("https://www.delfi.lv/"):
            continue

        # Exclude obvious non-articles
        if href.endswith("/comments"):
            continue
        if any(x in href for x in ["/kontakti", "/abonesana", "/podkasti", "/showroom"]):
            continue

        # Heuristic: Delfi articles usually contain a long numeric id segment (e.g., 1200xxxxx or 52xxxxx)
        if "/bizness/" in href and re.search(r"/\d{7,}/", href):
            urls.add(href)

    return sorted(urls)

def discover_n_urls(start_url: str, n: int, max_pages: int = 200) -> list[str]:
    collected = []
    seen = set()

    empty_pages_in_a_row = 0

    for page in range(1, max_pages + 1):
        page_url = build_page_url(start_url, page)
        r = SESSION.get(page_url, timeout=30)
        r.raise_for_status()

        page_urls = extract_listing_urls(r.text)

        added = 0
        for u in page_urls:
            if u not in seen:
                seen.add(u)
                collected.append(u)
                added += 1
                if len(collected) >= n:
                    return collected[:n]

        if added == 0:
            empty_pages_in_a_row += 1
        else:
            empty_pages_in_a_row = 0

        print(f"Page {page:03d}: found {len(page_urls)}; added {added}; total {len(collected)}")

        # stop if multiple consecutive pages add nothing (section end or layout changed)
        if empty_pages_in_a_row >= 5:
            break

        time.sleep(REQUEST_SLEEP)

    return collected[:n]

def extract_pub_date(soup: BeautifulSoup) -> str:
    # Try common meta tags first
    for key, val in [
        ("property", "article:published_time"),
        ("name", "article:published_time"),
        ("itemprop", "datePublished"),
    ]:
        tag = soup.find("meta", attrs={key: val})
        if tag and tag.get("content"):
            c = tag["content"].strip()
            # keep YYYY-MM-DD if present
            m = re.search(r"(\d{4}-\d{2}-\d{2})", c)
            if m:
                return m.group(1)

    # Fallback: look for date pattern in visible text near top
    text = soup.get_text(" ", strip=True)
    m = re.search(r"(\d{2}\.\d{2}\.\d{4})", text)
    if m:
        dd, mm, yyyy = m.group(1).split(".")
        return f"{yyyy}-{mm}-{dd}"

    return ""

def extract_title(soup: BeautifulSoup) -> str:
    h1 = soup.find("h1")
    if h1:
        return normalize_ws(h1.get_text(" ", strip=True))
    if soup.title:
        return normalize_ws(soup.title.get_text(" ", strip=True))
    return ""

def extract_full_text(soup: BeautifulSoup) -> str:
    # Prefer known content blocks (used in your Atmaskots scraping)
    blocks = soup.select("div.article-body__item--htmlElement")
    if blocks:
        parts = []
        for b in blocks:
            t = b.get_text(" ", strip=True)
            if t:
                parts.append(t)
        txt = normalize_ws(" ".join(parts))
    else:
        # Fallback: collect paragraphs inside <article> if present
        article = soup.find("article")
        if article:
            ps = article.find_all("p")
        else:
            ps = soup.find_all("p")

        parts = []
        for p in ps:
            t = p.get_text(" ", strip=True)
            if not t:
                continue
            # Skip typical footer/promotional boilerplate
            if t.startswith('Seko "Delfi"'):
                continue
            if "Publikācijas saturs" in t and "autortiesību" in t:
                continue
            parts.append(t)

        txt = normalize_ws(" ".join(parts))

    # Trim common tail boilerplate if it appears
    tail_markers = [
        'Seko "Delfi"',
        "Publikācijas saturs",
        "Vairāk lasi",
    ]
    for m in tail_markers:
        idx = txt.find(m)
        if idx != -1 and idx > 0:
            txt = normalize_ws(txt[:idx])
            break

    return txt

def scrape_article(url: str, topic: str, source: str, class_label: int) -> dict:
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    title = extract_title(soup)
    pub_date = extract_pub_date(soup)
    full_text = extract_full_text(soup)

    return {
        "title": title,
        "full_text": full_text,
        "source": source,
        "publication_date": pub_date,
        "topic": topic,
        "class_label": int(class_label),
        "url": url,
        "full_text_original": full_text,
        "text_length": len(full_text),
    }

all_dfs = []

for cat in CATEGORIES:
    print("\n======================================")
    print(f"Discovering URLs for: {cat['topic']}")
    urls = discover_n_urls(cat["start_url"], TARGET_PER_CATEGORY, max_pages=300)
    print(f"Discovered {len(urls)} URLs for {cat['topic']}")

    data = []
    for i, u in enumerate(urls, 1):
        try:
            print(f"Scraping {i}/{len(urls)}: {u}")
            rec = scrape_article(
                url=u,
                topic=cat["topic"],
                source=cat["source"],
                class_label=cat["class_label"],
            )
            data.append(rec)
        except Exception as e:
            print(f"Error: {u} -> {e}")
        time.sleep(REQUEST_SLEEP)

    df = pd.DataFrame(data, columns=[
        "title","full_text","source","publication_date","topic","class_label","url","full_text_original","text_length"
    ])

    out_name = f"delfi_bizness_{re.sub(r'[^a-z0-9]+','_', cat['topic'].lower()).strip('_')}.csv"
    df.to_csv(out_name, index=False, encoding="utf-8")
    print(f"Saved: {out_name} | rows: {len(df)}")

    all_dfs.append(df)

# Optional: merged (300 rows total)
merged = pd.concat(all_dfs, ignore_index=True)
merged.to_csv("delfi_reliable_raw.csv", index=False, encoding="utf-8")
print("\nSaved merged dataset: delfi_reliable_raw.csv")
print("Rows:", len(merged))
print("Per-topic counts:\n", merged["topic"].value_counts())
'''

with open("scraper.py", "w", encoding="utf-8") as f:
    f.write(code)

from google.colab import files
files.download("scraper.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(code[:500])

NameError: name 'code' is not defined

# 4.1. raw data inspection

In [ ]:
import pandas as pd
import re
from collections import Counter

FILE = "delfi_reliable_raw.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# -----------------------------
# TITLE INSPECTION
# -----------------------------
print("\nFIRST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).head(5), start=1):
        print(f"{i}. {title}")

print("\nLAST 5 TITLES:")
if "title" in df.columns:
    for i, title in enumerate(df["title"].fillna("").astype(str).tail(5), start=1):
        print(f"{i}. {title}")

# -----------------------------
# TEXT LENGTH STATS
# -----------------------------
print("\nTEXT LENGTH SUMMARY:")
if "full_text" in df.columns:
    lengths = df["full_text"].fillna("").astype(str).str.len()
    print("Min:", int(lengths.min()))
    print("Median:", int(lengths.median()))
    print("Max:", int(lengths.max()))

# -----------------------------
# REPEATED SENTENCE DETECTION
# -----------------------------
print("\nTOP 30 REPEATED SENTENCES:")

if "full_text" in df.columns:

    all_sentences = []

    for text in df["full_text"].fillna("").astype(str):
        sentences = re.split(r'(?<=[\.\!\?])\s+', text)

        for s in sentences:
            s = s.strip()
            if len(s) >= 40:
                all_sentences.append(s)

    sent_counts = pd.Series(all_sentences).value_counts().head(30)

    for sent, count in sent_counts.items():
        print(f"{count:>4}  {sent[:250]}")

# -----------------------------
# DELFI WORD CONTEXT ANALYSIS
# -----------------------------
print("\nDETECTING 'DELFI' CONTEXTS")

pattern = r'\b\S*delfi\S*\b'
contexts = []

for text in df["full_text"].fillna("").astype(str):

    matches = re.finditer(pattern, text, flags=re.IGNORECASE)

    for m in matches:
        start = max(m.start() - 40, 0)
        end = min(m.end() + 40, len(text))

        context = text[start:end]
        contexts.append(context.strip())

print("Total Delfi mentions:", len(contexts))

print("\nTOP 20 CONTEXTS AROUND 'DELFI':")

context_counts = Counter(contexts)

for context, count in context_counts.most_common(20):
    print(f"{count:>3}  {context}")

# -----------------------------
# DELFI PHRASE DETECTION
# -----------------------------
print("\nTOP PHRASES CONTAINING 'DELFI'")

phrases = []

for text in df["full_text"].fillna("").astype(str):

    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s]{0,25}delfi[A-Za-zĀ-Žā-ž0-9@\.\'\s]{0,25})',
        text,
        flags=re.IGNORECASE
    )

    for m in matches:
        phrases.append(m.strip())

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(20):
    print(f"{count:>3}  {phrase}")

ROWS: 600
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

FIRST 5 TITLES:
1. Ekonomika
2. Bargās ziemas rēķinu robi maciņā. Kas aktuāls par dzīves dārdzību noticis februārī?
3. Ziedu audzētāju izmaksas būtiski augušas – kas notiks ar cenām?
4. Degvielas piegādes gan Latvijā, gan Baltijā ir stabilas; izaicinājums ir cenu kāpums (19)
5. Naftas cenu kāpums nerimst; sasniegts divu gadu augstākais līmenis

LAST 5 TITLES:
1. Mediji: sankcijām pakļautie Krievijas produkti joprojām ieplūst ES (11)
2. Zviedrija atļauj Lietuvas un Polijas elektrības starpsavienojuma 'Harmony Link' izbūvi
3. Cenu kāpums kaimiņos: Igaunijā inflācija pērn sasniegusi 19,4%
4. Lietuvā pagājušā gada beigās palielinājies reģistrētā bezdarba līmenis
5. Beļģijas valdība un 'Engie' vienojas par divu reaktoru darbības pagarināšanu

TEXT LENGTH SUMMARY:
Min: 0
Median: 1774
Max: 19754

TOP 30 REPEATED SENTENCES:
  40  Lūdzu, izslēdz reklāmu bl

In [ ]:
%%writefile audit_delfi_reliable.py
from __future__ import annotations

import pandas as pd
import re
from collections import Counter

FILE = "delfi_reliable_dedup.csv"

df = pd.read_csv(FILE, encoding="utf-8")

print("ROWS:", len(df))
print("COLUMNS:", df.columns.tolist())

# --------------------------------------------------
# 1) Top repeated sentences
# --------------------------------------------------
print("\nTOP 30 REPEATED SENTENCES:")

all_sentences = []

for text in df["full_text_clean"].fillna("").astype(str):
    sentences = re.split(r'(?<=[\.\!\?])\s+', text)

    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) >= 40:
            all_sentences.append(sentence)

sentence_counts = Counter(all_sentences)

for sentence, count in sentence_counts.most_common(30):
    print(f"{count:>4}  {sentence[:250]}")

# --------------------------------------------------
# 2) Top contexts containing Delfi
# Finds: Delfi, "Delfi", Delfi.lv, Delfi Bizness, etc.
# --------------------------------------------------
print("\nTOP 30 CONTEXTS CONTAINING 'DELFI':")

delfi_pattern = r'\b\S*delfi\S*\b'
contexts = []

for text in df["full_text_clean"].fillna("").astype(str):
    for match in re.finditer(delfi_pattern, text, flags=re.IGNORECASE):
        start = max(match.start() - 40, 0)
        end = min(match.end() + 40, len(text))
        context = text[start:end].strip()
        contexts.append(context)

context_counts = Counter(contexts)

for context, count in context_counts.most_common(30):
    print(f"{count:>4}  {context}")

# --------------------------------------------------
# 3) Top phrases containing Delfi
# --------------------------------------------------
print("\nTOP 30 PHRASES CONTAINING 'DELFI':")

phrases = []

for text in df["full_text_clean"].fillna("").astype(str):
    matches = re.findall(
        r'([A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30}delfi[A-Za-zĀ-Žā-ž0-9"“”„\'\s@\.]{0,30})',
        text,
        flags=re.IGNORECASE
    )

    for match in matches:
        phrase = re.sub(r"\s+", " ", match).strip()
        if phrase:
            phrases.append(phrase)

phrase_counts = Counter(phrases)

for phrase, count in phrase_counts.most_common(30):
    print(f"{count:>4}  {phrase}")

# --------------------------------------------------
# 4) Count rows containing Delfi
# --------------------------------------------------
print("\nDELFI MENTION COUNTS:")

title_hits = df["title_clean"].fillna("").astype(str).str.contains(
    delfi_pattern, flags=re.IGNORECASE, regex=True
).sum()

text_hits = df["full_text_clean"].fillna("").astype(str).str.contains(
    delfi_pattern, flags=re.IGNORECASE, regex=True
).sum()

print("Delfi in title_clean:", int(title_hits))
print("Delfi in full_text_clean:", int(text_hits))

Writing audit_delfi_reliable.py


# 4.2 .data cleaning

In [ ]:
%%writefile clean_delfi_reliable.py
from __future__ import annotations

import re
import html
import pandas as pd

IN_FILE = "delfi_reliable_raw.csv"
OUT_FILE = "delfi_reliable_clean.csv"

MIN_TEXT_LEN = 300

# --------------------------------------------------
# Title cleanup:
# remove trailing comment counts like:
# "... ES (11)" -> "... ES"
# --------------------------------------------------
TITLE_TRAILING_COUNT_PATTERN = re.compile(r"\s*\(\d+\)\s*$")

# --------------------------------------------------
# Exact / near-exact repeated boilerplate sentences
# --------------------------------------------------
REMOVE_PATTERNS = [
    r"Lūdzu,\s*izslēdz\s*reklāmu\s*bloķētāju\s*vai\s*pārlādē\s*lapu\.?",
    r"Jautājumu\s*gadījumā\s*raksti\s*konts@delfi\.lv\.?",
    r"Pielāgojam\s*Tev\s*piemērotāko\s*abonēšanas\s*piedāvājumu\.{0,3}",
    r"PAR\s*PUBLIKĀCIJAS\s*SATURU\s*ATBILD\s*TĀS\s*AUTORI\.?",
    r"EIROPAS\s*SAVIENĪBA\s*NENOSAKA\s*PUBLIKĀCIJU\s*SATURU\s*UN\s*TO\s*IESPĒJAMO\s*TĀLĀKO\s*IZMANTOŠANU\.?",
    r"Par\s*šiem\s*un\s*citiem\s*jaunumiem\s*plašāk\s*lasi\s*apskatā\.?",
    r"Par\s*šiem\s*un\s*citiem\s*jaunumiem\s*lasāms\s*dienas\s*notikumu\s*apskatā\.?",
    r"pārlādē\s*lapu\.\s*Jautājumu\s*gadījumā\s*raksti\s*konts@delfi\.lv\.?",
    r"Laiks\s*ir\s*nauda,\s*tāpēc\s*[\"“”„']?\s*Delfi\s*Bizness\s*[\"“”„']?\s*piedāvā\s*kopsavilkumu\s*par\s*aktu(?:ā|a)l[^\.!\?]*",
]

# --------------------------------------------------
# Source / branding phrases to remove fully, with quotes if present
# --------------------------------------------------
SOURCE_PATTERNS = [
    # Existing patterns
    r'[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\s*Bizness\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\s*Biznes[s]?\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\.ee\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\s*Plus\s*[\"“”„\']',
    r'portāls\s*[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'portālu\s*[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'portālam\s*[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'Portāls\s*[\"“”„\']\s*Delfi\s*[\"“”„\']',
    r'\bDelfi\.lv\b',

    # New patterns from audit
    r'Re:Baltica\s+kopdarbs\s+ar\s+Delfi\s*\(Igaunija\)',
    r'[\"“”„\']\s*Delfi\s*Ärileht\s*[\"“”„\']',
    r'[\"“”„\']\s*Delfi\.lt\s*[\"“”„\']',
    r'[\"“”„\']\s*Spried\s+ar\s+Delfi\s*[\"“”„\']',
    r'\bSpried\s+ar\s+Delfi\b',
    r'\bDelfi\s+Bizness\b',
    r'\brus\.delfi\.ee\b',
    r'\bus\.delfi\.ee\b',

    # Broader source references that appeared in audit
    r'\bkā\s+raksta\s+[\"“”„\']\s*Delfi\.lt\s*[\"“”„\']',
    r'\bziņo\s+[\"“”„\']\s*Delfi\.lt\s*[\"“”„\']',
    r'\bvēsta\s+[\"“”„\']\s*Delfi\.lt\s*[\"“”„\']',
    r'\braksta\s+[\"“”„\']\s*Delfi\.lt\s*[\"“”„\']',
    r'\bportālam\s+[\"“”„\']\s*Delfi\s*Ärileht\s*[\"“”„\']',
    r'\bIgaunijas\s+portāls\s+rus\.delfi\.ee\b',
]

# Compile all regex patterns
COMPILED_REMOVE_PATTERNS = [
    re.compile(pat, flags=re.IGNORECASE | re.UNICODE | re.DOTALL)
    for pat in REMOVE_PATTERNS + SOURCE_PATTERNS
]


def normalize_text(text: object) -> str:
    """Basic technical cleanup: HTML, spaces, quotes, dashes."""
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Normalize quotes/dashes
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("\u00a0", " ").replace("\u200b", "")

    # Normalize spaces/newlines
    text = re.sub(r"\r\n|\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)

    return text.strip()


def clean_title(title: object) -> str:
    """Remove trailing comment counts in parentheses."""
    title = normalize_text(title)
    title = TITLE_TRAILING_COUNT_PATTERN.sub("", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def clean_full_text(text: object) -> str:
    """Remove repeated boilerplate and source/branding phrases."""
    text = normalize_text(text)

    for pattern in COMPILED_REMOVE_PATTERNS:
        text = pattern.sub(" ", text)

    # Clean punctuation leftovers
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?]){2,}", r"\1", text)
    text = re.sub(r"\s*-\s*-\s*", " - ", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)
    text = re.sub(r"\s+", " ", text)

    # Strip awkward leading punctuation after removals
    text = re.sub(r"^[,.;:\-\s]+", "", text)

    return text.strip()


# --------------------------------------------------
# Load
# --------------------------------------------------
df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)
columns_before = df.columns.tolist()

# --------------------------------------------------
# Create cleaned columns
# --------------------------------------------------
df["title_clean"] = df["title"].apply(clean_title)
df["full_text_clean"] = df["full_text"].apply(clean_full_text)
df["text_length_clean"] = df["full_text_clean"].fillna("").astype(str).str.len()

# --------------------------------------------------
# Remove empty / too short texts
# --------------------------------------------------
df = df[df["full_text_clean"].fillna("").astype(str).str.strip().str.len() >= MIN_TEXT_LEN].copy()

rows_after = len(df)
removed_rows = rows_before - rows_after

# Reset index
df = df.reset_index(drop=True)

# --------------------------------------------------
# Save
# --------------------------------------------------
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

# --------------------------------------------------
# Report
# --------------------------------------------------
columns_after = df.columns.tolist()
new_columns = [c for c in columns_after if c not in columns_before]

print("Saved:", OUT_FILE)
print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed rows:", removed_rows)

print("\nColumns before:")
print(columns_before)

print("\nColumns after:")
print(columns_after)

print("\nNew columns added:")
print(new_columns if new_columns else "None")

Writing clean_delfi_reliable.py


In [ ]:
!python clean_delfi_reliable.py

Saved: delfi_reliable_clean.csv
Rows before: 600
Rows after: 582
Removed rows: 18

Columns before:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length']

Columns after:
['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

New columns added:
['title_clean', 'full_text_clean', 'text_length_clean']


# 4.3. article deduplication

In [ ]:
%%writefile dedup_delfi_reliable.py
from __future__ import annotations

import pandas as pd

IN_FILE = "delfi_reliable_clean.csv"
OUT_FILE = "delfi_reliable_dedup.csv"

df = pd.read_csv(IN_FILE, encoding="utf-8")

rows_before = len(df)

# ----------------------------------------
# Remove fact-check articles
# ----------------------------------------
rows_before_filter = len(df)

df = df[
    ~df["title"].fillna("").astype(str).str.strip().str.lower().str.startswith("faktu pārbaude")
].copy()

rows_after_filter = len(df)

# ----------------------------------------
# Deduplicate by URL first
# ----------------------------------------
if "url" in df.columns:
    df["url"] = df["url"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["url"], keep="first")

rows_after_url = len(df)

# ----------------------------------------
# Deduplicate by cleaned text
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["full_text_clean"] = df["full_text_clean"].fillna("").astype(str).str.strip()
    df = df.drop_duplicates(subset=["full_text_clean"], keep="first")

rows_after_text = len(df)

# ----------------------------------------
# Recalculate clean text length
# ----------------------------------------
if "full_text_clean" in df.columns:
    df["text_length_clean"] = df["full_text_clean"].str.len()

df = df.reset_index(drop=True)
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print("Saved:", OUT_FILE)
print("Rows before processing:", rows_before)
print("Removed fact-check articles:", rows_before_filter - rows_after_filter)
print("Rows after URL dedup:", rows_after_url)
print("Rows after text dedup:", rows_after_text)
print("Total removed:", rows_before - len(df))

Writing dedup_delfi_reliable.py


In [ ]:
!python dedup_delfi_reliable.py

Saved: delfi_reliable_dedup.csv
Rows before processing: 582
Removed fact-check articles: 2
Rows after URL dedup: 577
Rows after text dedup: 577
Total removed: 5


# 4.4. audit

In [ ]:
!python audit_delfi_reliable.py

ROWS: 577
COLUMNS: ['title', 'full_text', 'source', 'publication_date', 'topic', 'class_label', 'url', 'full_text_original', 'text_length', 'title_clean', 'full_text_clean', 'text_length_clean']

TOP 30 REPEATED SENTENCES:
   4  "Rail Baltica" projekts paredz izveidot Eiropas standarta sliežu platuma dzelzceļa līniju no Tallinas līdz Lietuvas un Polijas robežai, lai tālāk ar dzelzceļu Baltijas valstis būtu iespējams savienot ar citām Eiropas valstīm.
   4  Baltijas valstīs plānots izbūvēt jaunu, 870 kilometru garu Eiropas sliežu platuma (1435 mm) dzelzceļa līniju ar vilcienu maksimālo ātrumu 240 kilometri stundā.
   3  Pasažieri tiek aicināti nekādā gadījumā nedoties uz lidostu, kamēr nav saņemts individuāls apstiprinājums par iekļaušanu konkrētā lidojuma pasažieru sarakstā un lidojumā.
   3  Tomēr, ņemot vērā strauji mainīgos apstākļus, reisa izpildē var tikt ieviestas neparedzētas izmaiņas.
   3  Šādos gadījumos pasažieri tiks savlaicīgi informēti.
   3  Kopējās projekta izmaksas atb